In [ ]:
"""| Feature                | TypedDict    | Pydantic   | JSON Schema        |
| ---------------------- | ------------ | ---------- | ------------------ |
| Ease of use            | ✅ Very easy  | ✅ Easy     | ⚠️ Medium (manual) |
| Validation             | ❌ None       | ✅ Strong   | ⚠️ Model-side only |
| Error handling         | ❌ Poor       | ✅ Built-in | ❌ Manual           |
| Type safety            | ⚠️ Weak      | ✅ Strong   | ⚠️ Depends         |
| Flexibility            | ❌ Low        | ⚠️ Medium  | ✅ Very high        |
| Production ready       | ❌ Not ideal  | ✅ Best     | ✅ Advanced use     |
| Debugging              | ❌ Hard       | ✅ Easy     | ⚠️ Medium          |
| Optional fields        | ⚠️ Confusing | ✅ Clean    | ✅ Explicit         |
| Nested structures      | ⚠️ Messy     | ✅ Clean    | ✅ Powerful         |
| Works with most models | ⚠️ Depends   | ✅ Better   | ✅ Best control     |
"""

In [ ]:
### USING TYPEDDICT
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated, Literal , Optional

load_dotenv()

## Why streaming dont work with structured_output ??
# Streaming is internally generated, but LangChain buffers the full output
# before parsing it into the structured schema, so callbacks don't receive tokens.

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
    
class Review(TypedDict):
    key_themes: Annotated[list[str], "Write down all the key themes."]
    pros: Annotated[Optional[list[str]], "Write all pros."]
    sentiment: Annotated[Literal['pos', 'neg'], "Return sentiment"]

## Annotated is used to give line to guide llms.
## Optional used means it can be generated or not according to the prompt
## Literal is used when you want output only in 'pos' , 'neg'

structured_model = llm.with_structured_output(Review)

result = structured_model.invoke("This phone has great battery life and performance but the camera is average.")

print(result) # {'key_themes': ['battery life', 'performance', 'camera'], 'pros': ['great battery life', 'great performance'], 'sentiment': 'pos'}

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'key_themes': ['battery life', 'performance', 'camera'], 'pros': ['great battery life', 'great performance'], 'sentiment': 'pos'}


In [ ]:
## USING Pydantic

from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from pydantic import BaseModel , Field
from typing import Optional , Literal

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
    
class Review(BaseModel):
    key_themes: list[str] = Field(description="Write all key themes")
    pros: Optional[list[str]] = Field(description="Write all pros.")
    sentiment: Literal['pos' , 'neg'] = Field(description='Return Sentiment')

structured_model = llm.with_structured_output(Review)

result = structured_model.invoke("This phone has great battery life and performance but the camera is average.")

print(result) # key_themes=['battery life', 'performance', 'camera'] pros=['great battery life', 'great performance'] sentiment='pos'

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


key_themes=['battery life', 'performance', 'camera'] pros=['great battery life', 'great performance'] sentiment='pos'


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# JSON Schema defines the STRUCTURE that the LLM must follow in its output
review_schema = {
    
    "type": "object",  
    # The overall output must be a JSON object (i.e., a dictionary in Python)

    "properties": {
        # "properties" defines all the fields (keys) that the object can contain

        "key_themes": {
            "type": "array",  
            # This field must be a list/array

            "items": {"type": "string"},  
            # Each element inside the list must be a string

            "description": "Write all key themes"  
            # Instruction to the LLM explaining what this field should contain
        },

        "pros": {
            "type": "array",  
            # This field must also be a list

            "items": {"type": "string"},  
            # Each item in the list must be a string

            "description": "Write all pros"  
            # Helps guide the model on what to extract
        },

        "sentiment": {
            "type": "string",  
            # This field must be a string value

            "enum": ["pos", "neg"],  
            # Restricts the output to ONLY these two values
            # Prevents the model from generating random text like "positive" or "good"

            "description": "Return sentiment"  
            # Instruction for the model to classify sentiment
        }
    },

    "required": ["key_themes", "sentiment"]  
    # These fields MUST be present in the output
    # If missing, the output is considered invalid
    # "pros" is not required, so it can be omitted if not found
}

structured_model = llm.with_structured_output(
    review_schema,
    method="json_schema"   # Important
)

result = structured_model.invoke(
    "This phone has great battery life and performance but the camera is average."
)

print(result)